# DichVideo Batch OmniVoice from SRT + Auto Sync Video

Upload many `.srt` files, one reference voice audio, and matching original video file(s). This notebook generates OmniVoice audio segments, then automatically runs Sync SRT + Audio Segments in Colab and downloads final synced video(s).

Use only with your own voice or with clear permission from the voice owner. Runtime > Change runtime type > GPU before running.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg

# OmniVoice README recommends torch/torchaudio 2.8.0 CUDA 12.8.
!pip -q install --force-reinstall --no-deps torch==2.8.0+cu128 torchaudio==2.8.0+cu128 torchvision==0.23.0+cu128 --index-url https://download.pytorch.org/whl/cu128
!pip -q install -U git+https://github.com/k2-fsa/OmniVoice.git faster-whisper soundfile


In [ ]:
# Use models already uploaded to Google Drive. This cell does not download model weights.
# Expected Drive folders:
# - MyDrive/models/OmniVoice
# - MyDrive/models/faster-whisper-large-v3
USE_DRIVE_MODELS = True
OMNIVOICE_MODEL_FOLDER = 'OmniVoice'
ASR_MODEL_FOLDER = 'faster-whisper-large-v3'

if USE_DRIVE_MODELS:
    from google.colab import drive
    from pathlib import Path
    import shutil

    drive.mount('/content/drive')
    drive_models_root = Path('/content/drive/MyDrive/models')
    local_models_root = Path('/content/models')

    def copy_drive_model(folder_name: str, required_file: str | None = None) -> str:
        drive_dir = drive_models_root / folder_name
        local_dir = local_models_root / folder_name
        if required_file and not (drive_dir / required_file).exists():
            raise FileNotFoundError(
                'Model not found in Google Drive. Upload the full model folder so this file exists: '
                f'{drive_dir / required_file}'
            )
        if not required_file and (not drive_dir.exists() or not any(drive_dir.iterdir())):
            raise FileNotFoundError(
                'Model folder not found in Google Drive. Upload the full model folder here: '
                f'{drive_dir}'
            )
        if not local_dir.exists():
            print(f'Copying {drive_dir} -> {local_dir}')
            local_dir.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(drive_dir, local_dir)
        else:
            print(f'Using local model copy: {local_dir}')
        return str(local_dir)

    OMNIVOICE_MODEL_PATH = copy_drive_model(OMNIVOICE_MODEL_FOLDER)
    ASR_MODEL_PATH = copy_drive_model(ASR_MODEL_FOLDER, required_file='model.bin')
else:
    OMNIVOICE_MODEL_PATH = 'k2-fsa/OmniVoice'
    ASR_MODEL_PATH = 'medium'

print('OMNIVOICE_MODEL_PATH =', OMNIVOICE_MODEL_PATH)
print('ASR_MODEL_PATH =', ASR_MODEL_PATH)


In [ ]:
from google.colab import files
from pathlib import Path
import shutil

SRT_DIR = Path('/content/dichvideo_srt_uploads')
VIDEO_DIR = Path('/content/dichvideo_video_uploads')
OUTPUT_DIR = Path('/content/dichvideo_omnivoice_audio')
SYNC_JOBS_DIR = Path('/content/dichvideo_sync_jobs')
FINAL_VIDEO_DIR = Path('/content/dichvideo_final_videos')

for folder in [SRT_DIR, OUTPUT_DIR, SYNC_JOBS_DIR, FINAL_VIDEO_DIR]:
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir(parents=True, exist_ok=True)
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

AUDIO_EXTENSIONS = {'.mp3', '.wav', '.m4a', '.flac', '.ogg'}
VIDEO_EXTENSIONS = {'.mp4', '.mov', '.mkv', '.webm', '.avi'}

print('Upload .srt file(s) and one reference voice audio file.')
uploaded = files.upload()
REF_AUDIO = None
for name, data in uploaded.items():
    suffix = Path(name).suffix.lower()
    if suffix == '.srt':
        (SRT_DIR / name).write_bytes(data)
    elif suffix in AUDIO_EXTENSIONS:
        ref_path = Path('/content') / name
        ref_path.write_bytes(data)
        REF_AUDIO = str(ref_path)
    elif suffix in VIDEO_EXTENSIONS:
        print(f'Skipped video in this cell: {name}. Upload videos in the next cell.')

if not REF_AUDIO:
    raise RuntimeError('Upload one reference audio file, e.g. audio-truyen.mp3')
if not any(SRT_DIR.glob('*.srt')):
    raise RuntimeError('Upload at least one .srt file.')

print('Reference audio:', REF_AUDIO)
print('SRT files:')
for path in sorted(SRT_DIR.glob('*.srt')):
    print('-', path.name)


In [ ]:
# Upload or replace original videos for auto Sync SRT + Audio Segments.
# You can rerun only this cell when you want to change 1.mp4, 2.mp4, 3.mp4.
from google.colab import files
from pathlib import Path

VIDEO_DIR = Path('/content/dichvideo_video_uploads')
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_EXTENSIONS = {'.mp4', '.mov', '.mkv', '.webm', '.avi'}
CLEAR_OLD_VIDEOS = True

if CLEAR_OLD_VIDEOS:
    for old_video in VIDEO_DIR.iterdir():
        if old_video.is_file() and old_video.suffix.lower() in VIDEO_EXTENSIONS:
            old_video.unlink()

print('Upload original video file(s), e.g. 1.mp4, 2.mp4, 3.mp4.')
uploaded_videos = files.upload()
for name, data in uploaded_videos.items():
    suffix = Path(name).suffix.lower()
    if suffix in VIDEO_EXTENSIONS:
        (VIDEO_DIR / name).write_bytes(data)
    else:
        print(f'Skipped non-video file: {name}')

video_files = sorted(path for path in VIDEO_DIR.iterdir() if path.suffix.lower() in VIDEO_EXTENSIONS)
if not video_files:
    print('No videos uploaded yet. Auto sync will ask again later if needed.')
else:
    print('Video files for auto sync:')
    for path in video_files:
        print('-', path.name)


In [ ]:
import html
import re
import subprocess
import torch
import unicodedata
from faster_whisper import WhisperModel

REFERENCE_WAV = '/content/ref.wav'
REFERENCE_START = '0'
REFERENCE_DURATION = '8'
LANGUAGE_ID = 'vi'
ASR_MODEL = ASR_MODEL_PATH

# Best-quality default: use 3-10 seconds of clean same-language speech, mono 24 kHz.
# If the source has leading silence, change REFERENCE_START to where speech begins.
subprocess.run([
    'ffmpeg', '-y', '-i', REF_AUDIO,
    '-ss', REFERENCE_START, '-t', REFERENCE_DURATION,
    '-vn', '-af', 'atrim=start=0,asetpts=PTS-STARTPTS,loudnorm=I=-18:TP=-2:LRA=11',
    '-ar', '24000', '-ac', '1',
    REFERENCE_WAV,
], check=True)

print('Reference WAV:', REFERENCE_WAV)

def clean_reference_text(text: str) -> str:
    text = html.unescape(unicodedata.normalize('NFKC', text))
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\{[^{}]*\}', ' ', text)
    text = re.sub(r'["“”‘’`]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if text and text[-1] not in '.,!?;:':
        text += '.'
    return text

device = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'int8'
asr = WhisperModel(ASR_MODEL, device=device, compute_type=compute_type)
segments, info = asr.transcribe(REFERENCE_WAV, language=LANGUAGE_ID, vad_filter=True)
REF_TEXT = clean_reference_text(' '.join(seg.text.strip() for seg in segments).strip())
print('REF_TEXT =', REF_TEXT)
if not REF_TEXT:
    raise RuntimeError('Whisper did not recognize text from the reference audio. Use a clearer reference clip.')


In [ ]:
# Embedded worker: creates grouped, trimmed segments/*.wav for safer OmniVoice batch TTS.
from pathlib import Path

WORKER_PATH = '/content/batch_omnivoice_from_srt.py'
Path(WORKER_PATH).write_text('from __future__ import annotations\n\nimport argparse\nimport html\nimport json\nimport logging\nimport re\nimport shutil\nimport subprocess\nimport time\nimport unicodedata\nfrom pathlib import Path\n\n\nMAX_GROUP_SEGMENTS = 5\nMAX_GROUP_GAP_SECONDS = 0.15\nDEFAULT_SEGMENT_TRIM_START_FRAMES = 4\nDEFAULT_SEGMENT_TRIM_END_FRAMES = 10\nDEFAULT_SEGMENT_TRIM_FPS = 30.0\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description="Batch OmniVoice TTS from uploaded SRT files.")\n    parser.add_argument("--srt-dir", required=True)\n    parser.add_argument("--output-dir", required=True)\n    parser.add_argument("--ref-audio", required=True)\n    parser.add_argument("--ref-text", default=None)\n    parser.add_argument("--model", default="k2-fsa/OmniVoice")\n    parser.add_argument("--device", default="cuda:0")\n    parser.add_argument("--dtype", default="float16", choices=["float16", "float32"])\n    parser.add_argument("--reference-start", type=float, default=0.0)\n    parser.add_argument("--reference-duration", type=float, default=8.0)\n    parser.add_argument("--speed", type=float, default=0.95)\n    parser.add_argument("--trim-start-seconds", type=float, default=0.0)\n    parser.add_argument("--end-padding-seconds", type=float, default=0.35)\n    parser.add_argument("--group-size", type=int, default=MAX_GROUP_SEGMENTS)\n    parser.add_argument("--max-group-gap-seconds", type=float, default=MAX_GROUP_GAP_SECONDS)\n    parser.add_argument("--segment-trim-start-frames", type=int, default=DEFAULT_SEGMENT_TRIM_START_FRAMES)\n    parser.add_argument("--segment-trim-end-frames", type=int, default=DEFAULT_SEGMENT_TRIM_END_FRAMES)\n    parser.add_argument("--segment-trim-fps", type=float, default=DEFAULT_SEGMENT_TRIM_FPS)\n    parser.add_argument("--num-step", type=int, default=32)\n    args = parser.parse_args()\n\n    srt_dir = Path(args.srt_dir)\n    output_dir = Path(args.output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    logger = _setup_logger(output_dir / "batch_omnivoice_from_srt.log")\n    _check_binary("ffmpeg")\n    _check_binary("ffprobe")\n\n    srt_files = sorted(srt_dir.glob("*.srt"))\n    if not srt_files:\n        raise RuntimeError(f"No .srt files found in {srt_dir}")\n\n    ref_audio = _prepare_reference_audio(\n        Path(args.ref_audio),\n        output_dir / "reference_24k.wav",\n        args.reference_start,\n        args.reference_duration,\n        logger,\n    )\n\n    import soundfile as sf\n    import torch\n    from omnivoice import OmniVoice\n\n    dtype = torch.float16 if args.dtype == "float16" else torch.float32\n    logger.info("Loading OmniVoice model=%s device=%s dtype=%s", args.model, args.device, args.dtype)\n    model = OmniVoice.from_pretrained(args.model, device_map=args.device, dtype=dtype)\n\n    for srt_index, srt_path in enumerate(srt_files, start=1):\n        logger.info("Processing SRT %s/%s: %s", srt_index, len(srt_files), srt_path)\n        segments = parse_srt(srt_path.read_text(encoding="utf-8-sig"))\n        if not segments:\n            logger.warning("Skipping empty SRT: %s", srt_path)\n            continue\n        _process_one_srt(\n            model=model,\n            sf=sf,\n            srt_path=srt_path,\n            segments=segments,\n            ref_audio=ref_audio,\n            ref_text=args.ref_text,\n            output_dir=output_dir,\n            speed=args.speed,\n            trim_start_seconds=args.trim_start_seconds,\n            end_padding_seconds=args.end_padding_seconds,\n            group_size=args.group_size,\n            max_group_gap_seconds=args.max_group_gap_seconds,\n            segment_trim_start_frames=args.segment_trim_start_frames,\n            segment_trim_end_frames=args.segment_trim_end_frames,\n            segment_trim_fps=args.segment_trim_fps,\n            num_step=args.num_step,\n            logger=logger,\n        )\n\n    zip_base = output_dir.parent / "omnivoice_audio_results"\n    if zip_base.with_suffix(".zip").exists():\n        zip_base.with_suffix(".zip").unlink()\n    shutil.make_archive(str(zip_base), "zip", output_dir)\n    logger.info("Created zip: %s.zip", zip_base)\n\n\ndef _process_one_srt(\n    model,\n    sf,\n    srt_path: Path,\n    segments: list[dict],\n    ref_audio: Path,\n    ref_text: str | None,\n    output_dir: Path,\n    speed: float,\n    trim_start_seconds: float,\n    end_padding_seconds: float,\n    group_size: int,\n    max_group_gap_seconds: float,\n    segment_trim_start_frames: int,\n    segment_trim_end_frames: int,\n    segment_trim_fps: float,\n    num_step: int,\n    logger: logging.Logger,\n) -> None:\n    name = _safe_stem(srt_path)\n    per_srt_dir = output_dir / name\n    raw_dir = per_srt_dir / "segments"\n    raw_dir.mkdir(parents=True, exist_ok=True)\n\n    grouped_segments = _group_adjacent_segments(\n        segments,\n        max_group_size=group_size,\n        max_gap_seconds=max_group_gap_seconds,\n    )\n    logger.info(\n        "Grouped %s SRT segment(s) into %s TTS segment(s), max_group_size=%s max_gap=%.3fs",\n        len(segments),\n        len(grouped_segments),\n        max(1, int(group_size)),\n        max_group_gap_seconds,\n    )\n\n    frame_fps = segment_trim_fps if segment_trim_fps > 0 else DEFAULT_SEGMENT_TRIM_FPS\n    trim_start_seconds = max(0.0, trim_start_seconds) + max(0, segment_trim_start_frames) / frame_fps\n    trim_end_seconds = max(0, segment_trim_end_frames) / frame_fps\n\n    scheduled = []\n    cursor = 0.0\n    for group_index, group in enumerate(grouped_segments, start=1):\n        original_text = " ".join(item["text"].strip() for item in group if item["text"].strip())\n        text = _prepare_tts_text(original_text)\n        first_segment = group[0]\n        last_segment = group[-1]\n        source_indexes = [item["index"] for item in group]\n        raw_wav = raw_dir / f"{group_index:04d}.wav"\n        logger.info(\n            "OmniVoice generate srt=%s group=%s source_indexes=%s chars=%s normalized_chars=%s trim_start=%.3fs trim_end=%.3fs",\n            srt_path.name,\n            group_index,\n            source_indexes,\n            len(original_text),\n            len(text),\n            trim_start_seconds,\n            trim_end_seconds,\n        )\n        audio = model.generate(\n            text=text,\n            ref_audio=str(ref_audio),\n            ref_text=ref_text,\n            speed=speed,\n            num_step=num_step,\n        )\n        audio_data = _trim_audio_edges(\n            audio[0],\n            sample_rate=24000,\n            trim_start_seconds=trim_start_seconds,\n            trim_end_seconds=trim_end_seconds,\n        )\n        sf.write(str(raw_wav), audio_data, 24000, subtype="PCM_24")\n        if end_padding_seconds > 0:\n            _pad_audio_tail(raw_wav, end_padding_seconds, logger)\n\n        raw_duration = _duration(raw_wav, logger)\n        scheduled_duration = raw_duration\n        scheduled_start = max(float(first_segment["start"]), cursor)\n        cursor = scheduled_start + scheduled_duration\n\n        scheduled.append({\n            "index": group_index,\n            "source_indexes": source_indexes,\n            "start": float(first_segment["start"]),\n            "end": float(last_segment["end"]),\n            "text": original_text,\n            "tts_text": text,\n            "scheduled_start": scheduled_start,\n            "scheduled_end": scheduled_start + scheduled_duration,\n            "audio_duration": scheduled_duration,\n            "audio": str(raw_wav.relative_to(per_srt_dir)),\n        })\n\n    full_wav = per_srt_dir / f"{name}_full.wav"\n    grouped_srt = per_srt_dir / f"{name}_grouped.srt"\n    _mix_scheduled_audio(scheduled, full_wav, logger, per_srt_dir)\n    _write_srt(grouped_srt, scheduled)\n    (per_srt_dir / f"{name}.srt").write_text(srt_path.read_text(encoding="utf-8-sig"), encoding="utf-8")\n    _write_json(per_srt_dir / "timing_schedule.json", scheduled)\n    logger.info("Finished %s -> %s; grouped SRT: %s", srt_path.name, full_wav, grouped_srt)\n\n\ndef _group_adjacent_segments(segments: list[dict], max_group_size: int, max_gap_seconds: float) -> list[list[dict]]:\n    max_group_size = max(1, int(max_group_size))\n    max_gap_seconds = max(0.0, float(max_gap_seconds))\n    groups = []\n    current = []\n    previous = None\n    for segment in segments:\n        if previous is None:\n            current = [segment]\n        else:\n            gap = float(segment["start"]) - float(previous["end"])\n            can_join = len(current) < max_group_size and gap <= max_gap_seconds\n            if can_join:\n                current.append(segment)\n            else:\n                groups.append(current)\n                current = [segment]\n        previous = segment\n    if current:\n        groups.append(current)\n    return groups\n\n\ndef parse_srt(content: str) -> list[dict]:\n    content = content.replace("\\r\\n", "\\n").replace("\\r", "\\n").strip()\n    if not content:\n        return []\n    blocks = re.split(r"\\n\\s*\\n", content)\n    segments = []\n    fallback_index = 1\n    for block in blocks:\n        lines = [line.strip() for line in block.split("\\n") if line.strip()]\n        if not lines:\n            continue\n        timing_line_index = next((i for i, line in enumerate(lines) if "-->" in line), None)\n        if timing_line_index is None:\n            continue\n        maybe_index = lines[0] if timing_line_index > 0 else str(fallback_index)\n        try:\n            index = int(re.sub(r"\\D+", "", maybe_index) or fallback_index)\n        except ValueError:\n            index = fallback_index\n        timing = lines[timing_line_index]\n        start_s, end_s = [part.strip().split()[0] for part in timing.split("-->", 1)]\n        text = " ".join(lines[timing_line_index + 1:]).strip()\n        if text:\n            segments.append({\n                "index": index,\n                "start": _parse_srt_timestamp(start_s),\n                "end": _parse_srt_timestamp(end_s),\n                "text": text,\n            })\n            fallback_index += 1\n    return segments\n\n\nCASE_SENSITIVE_READINGS = {\n    "AI": "ây ai",\n    "API": "ây pi ai",\n    "CPU": "si pi diu",\n    "GPU": "gi pi diu",\n    "CEO": "si i ô",\n    "USB": "diu ét bi",\n    "USD": "đô la Mỹ",\n    "USA": "Mỹ",\n    "UK": "Anh",\n    "OK": "ô kê",\n}\n\n\nLETTER_READINGS = {\n    "A": "ây", "B": "bi", "C": "si", "D": "đi", "E": "i", "F": "ép",\n    "G": "gi", "H": "hát", "I": "ai", "J": "giây", "K": "kây", "L": "eo",\n    "M": "em", "N": "en", "O": "ô", "P": "pi", "Q": "kiu", "R": "a",\n    "S": "ét", "T": "ti", "U": "diu", "V": "vi", "W": "đắp liu", "X": "ích",\n    "Y": "goai", "Z": "di",\n}\n\n\ndef _apply_pronunciation_dictionary(text: str) -> str:\n    for source, target in sorted(CASE_SENSITIVE_READINGS.items(), key=lambda item: len(item[0]), reverse=True):\n        text = re.sub(r"(?<![\\w])" + re.escape(source) + r"(?![\\w])", target, text)\n    return re.sub(\n        r"(?<![\\w])([A-Z]{2,})(?![\\w])",\n        lambda match: " ".join(LETTER_READINGS.get(char, char) for char in match.group(1)),\n        text,\n    )\n\n\ndef _prepare_tts_text(text: str) -> str:\n    text = html.unescape(unicodedata.normalize("NFKC", text))\n    text = re.sub(r"<[^>]+>", " ", text)\n    text = re.sub(r"\\{[^{}]*\\}", " ", text)\n    text = text.replace("…", ", ").replace("...", ", ")\n    replacements = {\n        "%": " phần trăm ",\n        "&": " và ",\n        "@": " a còng ",\n        "#": " số ",\n        "$": " đô la ",\n        "+": " cộng ",\n        "=": " bằng ",\n        "*": " sao ",\n        "×": " nhân ",\n        "÷": " chia ",\n        "/": " trên ",\n        "\\\\": " ",\n        "|": " ",\n        "_": " ",\n        "~": " ",\n        "^": " ",\n    }\n    for source, target in replacements.items():\n        text = text.replace(source, target)\n    text = _apply_pronunciation_dictionary(text)\n    text = re.sub(r"[\\[\\]{}()<>]", ", ", text)\n    text = re.sub(r"[\\"\'“”‘’`]+", "", text)\n    text = re.sub(r"\\s*[-–—]+\\s*", ", ", text)\n    text = re.sub(r"\\s+", " ", text).strip()\n    if text and text[-1] not in ".,!?;:":\n        text += "."\n    return text\n\n\ndef _parse_srt_timestamp(value: str) -> float:\n    match = re.match(r"(\\d+):(\\d+):(\\d+)[,.](\\d+)", value)\n    if not match:\n        raise ValueError(f"Invalid SRT timestamp: {value}")\n    h, m, s, ms = match.groups()\n    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms.ljust(3, "0")[:3]) / 1000\n\n\ndef _format_srt_timestamp(value: float) -> str:\n    value = max(0.0, float(value))\n    total_ms = int(round(value * 1000))\n    hours, remainder = divmod(total_ms, 3600_000)\n    minutes, remainder = divmod(remainder, 60_000)\n    seconds, milliseconds = divmod(remainder, 1000)\n    return f"{hours:02d}:{minutes:02d}:{seconds:02d},{milliseconds:03d}"\n\n\ndef _write_srt(path: Path, scheduled: list[dict]) -> None:\n    lines = []\n    for index, item in enumerate(scheduled, start=1):\n        lines.extend([\n            str(index),\n            f"{_format_srt_timestamp(item[\'start\'])} --> {_format_srt_timestamp(item[\'end\'])}",\n            item["text"],\n            "",\n        ])\n    path.write_text("\\n".join(lines), encoding="utf-8")\n\n\ndef _mix_scheduled_audio(scheduled: list[dict], output_path: Path, logger: logging.Logger, root_dir: Path) -> None:\n    total_duration = max(float(item["scheduled_end"]) for item in scheduled)\n    silence_path = root_dir / "_silence.wav"\n    _run(["ffmpeg", "-y", "-f", "lavfi", "-i", "anullsrc=channel_layout=mono:sample_rate=44100", "-t", f"{total_duration:.3f}", str(silence_path)], logger)\n    inputs = ["-i", str(silence_path)]\n    filters = []\n    mix_inputs = ["[0:a]"]\n    for input_index, item in enumerate(scheduled, start=1):\n        audio_path = root_dir / item["audio"]\n        inputs.extend(["-i", str(audio_path)])\n        delay_ms = max(0, int(float(item["scheduled_start"]) * 1000))\n        label = f"a{input_index}"\n        filters.append(f"[{input_index}:a]adelay={delay_ms}:all=1[{label}]")\n        mix_inputs.append(f"[{label}]")\n    filter_complex = ";".join(filters + [f"{\'\'.join(mix_inputs)}amix=inputs={len(mix_inputs)}:normalize=0[out]"])\n    _run(["ffmpeg", "-y", *inputs, "-filter_complex", filter_complex, "-map", "[out]", "-ac", "2", "-ar", "44100", "-c:a", "pcm_s24le", str(output_path)], logger)\n    silence_path.unlink(missing_ok=True)\n\n\ndef _prepare_reference_audio(input_path: Path, output_path: Path, start: float, duration: float, logger: logging.Logger) -> Path:\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    _run([\n        "ffmpeg", "-y", "-i", str(input_path),\n        "-ss", f"{start:.3f}", "-t", f"{duration:.3f}",\n        "-vn", "-af", "atrim=start=0,asetpts=PTS-STARTPTS,loudnorm=I=-18:TP=-2:LRA=11",\n        "-ac", "1", "-ar", "24000", "-c:a", "pcm_s24le",\n        str(output_path),\n    ], logger)\n    return output_path\n\n\ndef _trim_audio_edges(audio_data, sample_rate: int, trim_start_seconds: float, trim_end_seconds: float):\n    start_samples = max(0, int(round(sample_rate * trim_start_seconds)))\n    end_samples = max(0, int(round(sample_rate * trim_end_seconds)))\n    if start_samples <= 0 and end_samples <= 0:\n        return audio_data\n    if len(audio_data) <= start_samples + end_samples:\n        return audio_data\n    end_index = len(audio_data) - end_samples if end_samples > 0 else len(audio_data)\n    return audio_data[start_samples:end_index]\n\n\ndef _pad_audio_tail(path: Path, pad_seconds: float, logger: logging.Logger) -> None:\n    temp_path = path.with_name(path.stem + "_pad" + path.suffix)\n    _run([\n        "ffmpeg", "-y", "-i", str(path),\n        "-filter:a", f"apad=pad_dur={pad_seconds:.3f}",\n        "-ac", "1", "-ar", "24000", "-c:a", "pcm_s24le",\n        str(temp_path),\n    ], logger)\n    temp_path.replace(path)\n\n\ndef _duration(path: Path, logger: logging.Logger) -> float:\n    completed = _run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", str(path)], logger)\n    return float(completed.stdout.strip())\n\n\ndef _safe_stem(path: Path) -> str:\n    stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", path.stem).strip("._")\n    return stem or "srt"\n\n\ndef _run(cmd: list[str], logger: logging.Logger) -> subprocess.CompletedProcess:\n    logger.info("Running command: %s", " ".join(cmd))\n    completed = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")\n    if completed.stdout.strip():\n        logger.info("stdout: %s", completed.stdout.strip()[-3000:])\n    if completed.stderr.strip():\n        logger.info("stderr: %s", completed.stderr.strip()[-3000:])\n    if completed.returncode != 0:\n        raise RuntimeError(f"Command failed with code {completed.returncode}: {\' \'.join(cmd)}")\n    return completed\n\n\ndef _check_binary(name: str) -> None:\n    if shutil.which(name) is None:\n        raise RuntimeError(f"Missing dependency: {name}")\n\n\ndef _write_json(path: Path, data) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")\n\n\ndef _setup_logger(log_path: Path) -> logging.Logger:\n    logger = logging.getLogger("dichvideo_batch_omnivoice_from_srt")\n    logger.setLevel(logging.INFO)\n    logger.handlers.clear()\n    formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")\n    file_handler = logging.FileHandler(log_path, encoding="utf-8")\n    file_handler.setFormatter(formatter)\n    logger.addHandler(file_handler)\n    stream_handler = logging.StreamHandler()\n    stream_handler.setFormatter(formatter)\n    logger.addHandler(stream_handler)\n    return logger\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('Worker written:', WORKER_PATH)


In [ ]:
import torch

MODEL = OMNIVOICE_MODEL_PATH
NUM_STEP = '64'  # Higher quality/stability for final output; use 32 only when you need faster output.
SPEED = '1.00'  # Slightly slower helps OmniVoice pronounce Vietnamese and punctuation-heavy text more reliably.
TRIM_START_SECONDS = '0.0'  # Avoid cutting the first phoneme; raise only if your model adds a fixed click/silence.
END_PADDING_SECONDS = '0.0'  # Do not add silence after frame-based end trim.
GROUP_SIZE = '5'
MAX_GROUP_GAP_SECONDS = '0.15'
SEGMENT_TRIM_START_FRAMES = '4'
SEGMENT_TRIM_END_FRAMES = '8'
SEGMENT_TRIM_FPS = '30'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
DTYPE = 'float16' if torch.cuda.is_available() else 'float32'

!python "$WORKER_PATH" \
  --srt-dir "$SRT_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --ref-audio "$REFERENCE_WAV" \
  --ref-text "$REF_TEXT" \
  --reference-start "0" \
  --reference-duration "9999" \
  --model "$MODEL" \
  --device "$DEVICE" \
  --dtype "$DTYPE" \
  --speed "$SPEED" \
  --trim-start-seconds "$TRIM_START_SECONDS" \
  --end-padding-seconds "$END_PADDING_SECONDS" \
  --group-size "$GROUP_SIZE" \
  --max-group-gap-seconds "$MAX_GROUP_GAP_SECONDS" \
  --segment-trim-start-frames "$SEGMENT_TRIM_START_FRAMES" \
  --segment-trim-end-frames "$SEGMENT_TRIM_END_FRAMES" \
  --segment-trim-fps "$SEGMENT_TRIM_FPS" \
  --num-step "$NUM_STEP"


In [ ]:
# Embedded local Sync SRT + Audio Segments helper from dichvideo/segment_retimer.py
from pathlib import Path

PACKAGE_DIR = Path('/content/dichvideo')
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
(PACKAGE_DIR / '__init__.py').write_text('', encoding='utf-8')
(PACKAGE_DIR / 'segment_retimer.py').write_text('from __future__ import annotations\n\nimport json\nimport logging\nimport re\nimport shutil\nimport subprocess\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Iterable\n\n\n@dataclass\nclass SegmentRetimeConfig:\n    min_video_ratio: float = 0.70\n    max_video_ratio: float = 1.40\n    min_audio_tempo: float = 0.75\n    max_audio_tempo: float = 1.80\n    keep_gaps: bool = True\n    original_audio_volume: float = 0.0\n    crf: int = 20\n    preset: str = "veryfast"\n    output_fps: int = 30\n\n\n@dataclass\nclass SegmentRetimeUpdate:\n    status: str\n    log_text: str\n    video_path: str | None = None\n    schedule_path: str | None = None\n    log_path: str | None = None\n    job_path: str | None = None\n\n\n@dataclass\nclass VideoTimelinePiece:\n    start: float\n    end: float\n    ratio: float\n    duration: float\n\n\ndef retime_video_to_audio_segments(\n    video_path: Path,\n    srt_path: Path,\n    audio_paths: list[Path],\n    jobs_root: Path,\n    config: SegmentRetimeConfig,\n) -> Iterable[SegmentRetimeUpdate]:\n    job_dir = _create_job_dir(jobs_root)\n    log_path = job_dir / "logs" / "segment_retimer.log"\n    logger = _setup_logger(log_path)\n\n    def update(message: str, video: Path | None = None, schedule: Path | None = None) -> SegmentRetimeUpdate:\n        logger.info(message)\n        return SegmentRetimeUpdate(\n            status=message,\n            log_text=_tail(log_path),\n            video_path=str(video) if video and video.exists() else None,\n            schedule_path=str(schedule) if schedule and schedule.exists() else None,\n            log_path=str(log_path),\n            job_path=str(job_dir),\n        )\n\n    try:\n        _check_binary("ffmpeg")\n        _check_binary("ffprobe")\n        if not video_path.exists():\n            raise RuntimeError(f"Video not found: {video_path}")\n        if not srt_path.exists():\n            raise RuntimeError(f"SRT not found: {srt_path}")\n        if not audio_paths:\n            raise RuntimeError("No audio segment files provided.")\n\n        input_video = job_dir / "input" / video_path.name\n        input_srt = job_dir / "input" / srt_path.name\n        shutil.copy2(video_path, input_video)\n        shutil.copy2(srt_path, input_srt)\n        copied_audio_paths = []\n        for index, audio_path in enumerate(_sort_audio_paths(audio_paths), start=1):\n            copied = job_dir / "input" / "audio_segments" / f"{index:04d}{audio_path.suffix.lower() or \'.wav\'}"\n            copied.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(audio_path, copied)\n            copied_audio_paths.append(copied)\n\n        yield update(f"Created segment retime job: {job_dir}")\n\n        segments = parse_srt(input_srt.read_text(encoding="utf-8-sig"))\n        if len(copied_audio_paths) < len(segments):\n            raise RuntimeError(f"Need at least {len(segments)} audio segments, got {len(copied_audio_paths)}.")\n        if len(copied_audio_paths) > len(segments):\n            logger.warning("More audio files than SRT segments. Extra files will be ignored: %s > %s", len(copied_audio_paths), len(segments))\n            copied_audio_paths = copied_audio_paths[:len(segments)]\n\n        video_duration = _duration(input_video, logger)\n        _validate_segments(segments, video_duration)\n        has_original_audio = config.original_audio_volume > 0 and _has_audio(input_video, logger)\n        if config.original_audio_volume > 0 and not has_original_audio:\n            logger.warning("Original audio volume was requested, but source video has no readable audio stream.")\n        yield update(f"Loaded {len(segments)} SRT segment(s). Video duration: {video_duration:.3f}s")\n\n        video_timeline = []\n        audio_piece_list = []\n        original_audio_piece_list = []\n        schedule = []\n        cursor = 0.0\n        previous_end = 0.0\n\n        for index, (segment, audio_path) in enumerate(zip(segments, copied_audio_paths), start=1):\n            if config.keep_gaps and segment["start"] - previous_end >= 0.05:\n                gap_duration = segment["start"] - previous_end\n                gap_audio = job_dir / "work" / "audio_pieces" / f"{len(audio_piece_list):05d}_gap.wav"\n                video_timeline.append(VideoTimelinePiece(previous_end, segment["start"], 1.0, gap_duration))\n                _make_silence(gap_audio, gap_duration, logger)\n                audio_piece_list.append(gap_audio)\n                if has_original_audio:\n                    original_gap_audio = job_dir / "work" / "original_audio_pieces" / f"{len(original_audio_piece_list):05d}_gap.wav"\n                    _extract_original_audio_piece(input_video, previous_end, segment["start"], 1.0, gap_duration, original_gap_audio, logger)\n                    original_audio_piece_list.append(original_gap_audio)\n                schedule.append({\n                    "type": "gap",\n                    "original_start": round(previous_end, 3),\n                    "original_end": round(segment["start"], 3),\n                    "output_start": round(cursor, 3),\n                    "output_end": round(cursor + gap_duration, 3),\n                    "duration": round(gap_duration, 3),\n                })\n                cursor += gap_duration\n\n            original_video_duration = max(0.001, segment["end"] - segment["start"])\n            original_audio_duration = _duration(audio_path, logger)\n            desired_video_ratio = original_audio_duration / original_video_duration\n            video_ratio = _clamp(desired_video_ratio, config.min_video_ratio, config.max_video_ratio)\n            target_duration = original_video_duration * video_ratio\n\n            if abs(target_duration - original_audio_duration) <= 0.03:\n                audio_tempo = 1.0\n            else:\n                audio_tempo = _clamp(original_audio_duration / target_duration, config.min_audio_tempo, config.max_audio_tempo)\n\n            audio_piece = job_dir / "work" / "audio_pieces" / f"{len(audio_piece_list):05d}_seg_{index:04d}.wav"\n            video_timeline.append(VideoTimelinePiece(segment["start"], segment["end"], video_ratio, target_duration))\n            _retime_audio_piece(audio_path, audio_tempo, target_duration, audio_piece, logger)\n            actual_audio_piece_duration = _duration(audio_piece, logger)\n            if abs(target_duration - actual_audio_piece_duration) > 0.08:\n                logger.warning(\n                    "Rendered segment duration mismatch index=%s target=%.3fs video_piece=%.3fs audio_piece=%.3fs",\n                    index,\n                    target_duration,\n                    target_duration,\n                    actual_audio_piece_duration,\n                )\n            if has_original_audio:\n                original_audio_piece = job_dir / "work" / "original_audio_pieces" / f"{len(original_audio_piece_list):05d}_seg_{index:04d}.wav"\n                _extract_original_audio_piece(\n                    input_video,\n                    segment["start"],\n                    segment["end"],\n                    1.0 / video_ratio,\n                    target_duration,\n                    original_audio_piece,\n                    logger,\n                )\n                original_audio_piece_list.append(original_audio_piece)\n\n            audio_piece_list.append(audio_piece)\n            schedule.append({\n                "type": "segment",\n                "index": segment["index"],\n                "text": segment["text"],\n                "original_start": round(segment["start"], 3),\n                "original_end": round(segment["end"], 3),\n                "original_video_duration": round(original_video_duration, 3),\n                "original_audio_duration": round(original_audio_duration, 3),\n                "desired_video_ratio": round(desired_video_ratio, 5),\n                "video_ratio": round(video_ratio, 5),\n                "target_duration": round(target_duration, 3),\n                "audio_tempo": round(audio_tempo, 5),\n                "actual_video_piece_duration": round(target_duration, 3),\n                "actual_audio_piece_duration": round(actual_audio_piece_duration, 3),\n                "output_start": round(cursor, 3),\n                "output_end": round(cursor + target_duration, 3),\n                "audio_file": str(audio_path.name),\n            })\n            logger.info(\n                "Segment %s video=%.3fs audio=%.3fs desired_ratio=%.5f video_ratio=%.5f target=%.3fs audio_tempo=%.5f",\n                index,\n                original_video_duration,\n                original_audio_duration,\n                desired_video_ratio,\n                video_ratio,\n                target_duration,\n                audio_tempo,\n            )\n            cursor += target_duration\n            previous_end = segment["end"]\n            yield update(f"Processed segment {index}/{len(segments)}")\n\n        if config.keep_gaps and video_duration - previous_end >= 0.05:\n            gap_duration = video_duration - previous_end\n            gap_audio = job_dir / "work" / "audio_pieces" / f"{len(audio_piece_list):05d}_tail.wav"\n            video_timeline.append(VideoTimelinePiece(previous_end, video_duration, 1.0, gap_duration))\n            _make_silence(gap_audio, gap_duration, logger)\n            audio_piece_list.append(gap_audio)\n            if has_original_audio:\n                original_tail_audio = job_dir / "work" / "original_audio_pieces" / f"{len(original_audio_piece_list):05d}_tail.wav"\n                _extract_original_audio_piece(input_video, previous_end, video_duration, 1.0, gap_duration, original_tail_audio, logger)\n                original_audio_piece_list.append(original_tail_audio)\n            schedule.append({\n                "type": "tail",\n                "original_start": round(previous_end, 3),\n                "original_end": round(video_duration, 3),\n                "output_start": round(cursor, 3),\n                "output_end": round(cursor + gap_duration, 3),\n                "duration": round(gap_duration, 3),\n            })\n            cursor += gap_duration\n\n        schedule_path = job_dir / "output" / "retime_schedule.json"\n        _write_json(schedule_path, {\n            "source_video": str(input_video),\n            "source_srt": str(input_srt),\n            "config": config.__dict__,\n            "output_duration": round(cursor, 3),\n            "items": schedule,\n        })\n\n        yield update("Rendering retimed video timeline and concatenating audio...")\n        retimed_video = job_dir / "output" / "retimed_video.mp4"\n        retimed_audio = job_dir / "output" / "retimed_audio.wav"\n        _render_video_timeline(input_video, video_timeline, retimed_video, config, logger, job_dir)\n        _concat_media(audio_piece_list, retimed_audio, "audio", logger, job_dir)\n        audio_for_mux = retimed_audio\n        if has_original_audio:\n            original_retimed_audio = job_dir / "output" / "original_retimed_audio.wav"\n            mixed_audio = job_dir / "output" / "mixed_audio.wav"\n            _concat_media(original_audio_piece_list, original_retimed_audio, "original_audio", logger, job_dir)\n            _mix_audio(original_retimed_audio, retimed_audio, mixed_audio, config.original_audio_volume, logger)\n            audio_for_mux = mixed_audio\n\n        final_video = job_dir / "output" / "final.mp4"\n        _run([\n            "ffmpeg", "-y",\n            "-i", str(retimed_video),\n            "-i", str(audio_for_mux),\n            "-map", "0:v:0",\n            "-map", "1:a:0",\n            "-c:v", "copy",\n            "-c:a", "aac",\n            "-shortest",\n            str(final_video),\n        ], logger)\n        yield update("Done", video=final_video, schedule=schedule_path)\n    except Exception:\n        logger.exception("Segment retime failed")\n        yield SegmentRetimeUpdate(\n            status="Segment retime failed. Xem log de biet chi tiet.",\n            log_text=_tail(log_path),\n            log_path=str(log_path),\n            job_path=str(job_dir),\n        )\n\n\ndef parse_srt(content: str) -> list[dict]:\n    content = content.replace("\\r\\n", "\\n").replace("\\r", "\\n").strip()\n    if not content:\n        return []\n    blocks = re.split(r"\\n\\s*\\n", content)\n    segments = []\n    fallback_index = 1\n    for block in blocks:\n        lines = [line.strip() for line in block.split("\\n") if line.strip()]\n        if not lines:\n            continue\n        timing_line_index = next((i for i, line in enumerate(lines) if "-->" in line), None)\n        if timing_line_index is None:\n            continue\n        maybe_index = lines[0] if timing_line_index > 0 else str(fallback_index)\n        index = int(re.sub(r"\\D+", "", maybe_index) or fallback_index)\n        timing = lines[timing_line_index]\n        start_s, end_s = [part.strip().split()[0] for part in timing.split("-->", 1)]\n        text = " ".join(lines[timing_line_index + 1:]).strip()\n        if text:\n            segments.append({\n                "index": index,\n                "start": _parse_srt_timestamp(start_s),\n                "end": _parse_srt_timestamp(end_s),\n                "text": text,\n            })\n            fallback_index += 1\n    return segments\n\n\ndef _parse_srt_timestamp(value: str) -> float:\n    match = re.match(r"(\\d+):(\\d+):(\\d+)[,.](\\d+)", value)\n    if not match:\n        raise ValueError(f"Invalid SRT timestamp: {value}")\n    h, m, s, ms = match.groups()\n    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms.ljust(3, "0")[:3]) / 1000\n\n\ndef _extract_video_piece(input_video: Path, start: float, end: float, output_path: Path, config: SegmentRetimeConfig, logger: logging.Logger) -> None:\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    duration = max(0.001, end - start)\n    video_filter = (\n        f"trim=start={start:.6f}:end={end:.6f},"\n        "setpts=PTS-STARTPTS,"\n        f"fps={config.output_fps},"\n        "tpad=stop_mode=clone:stop_duration=1,"\n        f"trim=duration={duration:.6f},"\n        "setpts=PTS-STARTPTS"\n    )\n    _run([\n        "ffmpeg", "-y",\n        "-i", str(input_video),\n        "-an",\n        "-filter:v", video_filter,\n        "-c:v", "libx264",\n        "-preset", config.preset,\n        "-crf", str(config.crf),\n        "-pix_fmt", "yuv420p",\n        str(output_path),\n    ], logger)\n\n\ndef _extract_and_retime_video_piece(input_video: Path, start: float, end: float, ratio: float, target_duration: float, output_path: Path, config: SegmentRetimeConfig, logger: logging.Logger) -> None:\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    target_duration = max(0.001, target_duration)\n    video_filter = (\n        f"trim=start={start:.6f}:end={end:.6f},"\n        "setpts=PTS-STARTPTS,"\n        f"setpts={ratio:.8f}*PTS,"\n        f"fps={config.output_fps},"\n        "tpad=stop_mode=clone:stop_duration=1,"\n        f"trim=duration={target_duration:.6f},"\n        "setpts=PTS-STARTPTS"\n    )\n    _run([\n        "ffmpeg", "-y",\n        "-i", str(input_video),\n        "-an",\n        "-filter:v", video_filter,\n        "-c:v", "libx264",\n        "-preset", config.preset,\n        "-crf", str(config.crf),\n        "-pix_fmt", "yuv420p",\n        str(output_path),\n    ], logger)\n\n\ndef _render_video_timeline(\n    input_video: Path,\n    pieces: list[VideoTimelinePiece],\n    output_path: Path,\n    config: SegmentRetimeConfig,\n    logger: logging.Logger,\n    job_dir: Path,\n) -> None:\n    if not pieces:\n        raise RuntimeError("No video timeline pieces to render.")\n\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    filter_script_path = job_dir / "work" / "video_timeline_filter.txt"\n    filter_script_path.parent.mkdir(parents=True, exist_ok=True)\n\n    lines = []\n    labels = []\n    for index, piece in enumerate(pieces):\n        duration = max(0.001, piece.duration)\n        label = f"v{index}"\n        labels.append(f"[{label}]")\n        lines.append(\n            f"[0:v]trim=start={piece.start:.6f}:end={piece.end:.6f},"\n            "setpts=PTS-STARTPTS,"\n            f"setpts={piece.ratio:.8f}*PTS,"\n            f"fps={config.output_fps},"\n            "tpad=stop_mode=clone:stop_duration=1,"\n            f"trim=duration={duration:.6f},"\n            f"setpts=PTS-STARTPTS[{label}]"\n        )\n\n    lines.append("".join(labels) + f"concat=n={len(pieces)}:v=1:a=0[vout]")\n    filter_script_path.write_text(";\\n".join(lines), encoding="utf-8")\n\n    _run([\n        "ffmpeg", "-y",\n        "-i", str(input_video),\n        "-filter_complex_script", str(filter_script_path),\n        "-map", "[vout]",\n        "-an",\n        "-c:v", "libx264",\n        "-preset", config.preset,\n        "-crf", str(config.crf),\n        "-pix_fmt", "yuv420p",\n        str(output_path),\n    ], logger)\n\n\ndef _retime_audio_piece(input_audio: Path, tempo: float, target_duration: float, output_path: Path, logger: logging.Logger) -> None:\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    filters = []\n    if abs(tempo - 1.0) > 0.001:\n        filters.append(_atempo_filter(tempo))\n    filters.extend(["apad", f"atrim=0:{target_duration:.3f}"])\n    _run([\n        "ffmpeg", "-y",\n        "-i", str(input_audio),\n        "-filter:a", ",".join(filters),\n        "-ac", "2",\n        "-ar", "44100",\n        str(output_path),\n    ], logger)\n\n\ndef _extract_original_audio_piece(input_video: Path, start: float, end: float, tempo: float, target_duration: float, output_path: Path, logger: logging.Logger) -> None:\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    target_duration = max(0.001, target_duration)\n    filters = [f"atrim=start={start:.6f}:end={end:.6f}", "asetpts=PTS-STARTPTS"]\n    if abs(tempo - 1.0) > 0.001:\n        filters.append(_atempo_filter(tempo))\n    filters.extend(["apad", f"atrim=0:{target_duration:.6f}", "asetpts=PTS-STARTPTS"])\n    _run([\n        "ffmpeg", "-y",\n        "-i", str(input_video),\n        "-vn",\n        "-filter:a", ",".join(filters),\n        "-ac", "2",\n        "-ar", "44100",\n        str(output_path),\n    ], logger)\n\n\ndef _make_silence(output_path: Path, duration: float, logger: logging.Logger) -> None:\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    _run([\n        "ffmpeg", "-y",\n        "-f", "lavfi",\n        "-i", "anullsrc=channel_layout=stereo:sample_rate=44100",\n        "-t", f"{duration:.3f}",\n        str(output_path),\n    ], logger)\n\n\ndef _concat_media(paths: list[Path], output_path: Path, kind: str, logger: logging.Logger, job_dir: Path) -> None:\n    list_path = job_dir / "work" / f"{kind}_concat.txt"\n    list_path.parent.mkdir(parents=True, exist_ok=True)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    list_path.write_text("".join(f"file \'{path.resolve().as_posix()}\'\\n" for path in paths), encoding="utf-8")\n    if kind == "video":\n        cmd = ["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(list_path), "-c:v", "libx264", "-preset", "veryfast", "-crf", "20", "-pix_fmt", "yuv420p", str(output_path)]\n    else:\n        cmd = ["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(list_path), "-c:a", "pcm_s16le", "-ac", "2", "-ar", "44100", str(output_path)]\n    _run(cmd, logger)\n\n\ndef _mix_audio(original_audio: Path, dubbed_audio: Path, output_path: Path, original_volume: float, logger: logging.Logger) -> None:\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    volume = max(0.0, float(original_volume))\n    _run([\n        "ffmpeg", "-y",\n        "-i", str(original_audio),\n        "-i", str(dubbed_audio),\n        "-filter_complex", f"[0:a]volume={volume:.5f}[a0];[a0][1:a]amix=inputs=2:normalize=0[out]",\n        "-map", "[out]",\n        "-ac", "2",\n        "-ar", "44100",\n        str(output_path),\n    ], logger)\n\n\ndef _atempo_filter(tempo: float) -> str:\n    parts = []\n    remaining = tempo\n    while remaining > 2.0:\n        parts.append("atempo=2.0")\n        remaining /= 2.0\n    while remaining < 0.5:\n        parts.append("atempo=0.5")\n        remaining /= 0.5\n    parts.append(f"atempo={remaining:.5f}")\n    return ",".join(parts)\n\n\ndef _sort_audio_paths(paths: list[Path]) -> list[Path]:\n    def key(path: Path):\n        numbers = re.findall(r"\\d+", path.stem)\n        return (int(numbers[-1]) if numbers else 10**9, path.name.lower())\n    return sorted(paths, key=key)\n\n\ndef _validate_segments(segments: list[dict], video_duration: float) -> None:\n    previous_end = 0.0\n    for index, segment in enumerate(segments, start=1):\n        start = float(segment["start"])\n        end = float(segment["end"])\n        if end <= start:\n            raise RuntimeError(\n                f"SRT segment {index} has invalid timing: start={_format_seconds(start)}, end={_format_seconds(end)}."\n            )\n        if start < previous_end - 0.001:\n            raise RuntimeError(\n                f"SRT segment {index} starts before the previous segment ends: "\n                f"previous_end={_format_seconds(previous_end)}, start={_format_seconds(start)}."\n            )\n        if end > video_duration + 0.25:\n            raise RuntimeError(\n                f"SRT segment {index} ends after the video duration: "\n                f"end={_format_seconds(end)}, video_duration={_format_seconds(video_duration)}."\n            )\n        previous_end = max(previous_end, end)\n\n\ndef _format_seconds(value: float) -> str:\n    return f"{value:.3f}s"\n\n\ndef _duration(path: Path, logger: logging.Logger) -> float:\n    completed = _run([\n        "ffprobe", "-v", "error",\n        "-show_entries", "format=duration",\n        "-of", "default=noprint_wrappers=1:nokey=1",\n        str(path),\n    ], logger)\n    return float(completed.stdout.strip())\n\n\ndef _has_audio(path: Path, logger: logging.Logger) -> bool:\n    completed = _run([\n        "ffprobe", "-v", "error",\n        "-select_streams", "a:0",\n        "-show_entries", "stream=index",\n        "-of", "csv=p=0",\n        str(path),\n    ], logger)\n    return bool(completed.stdout.strip())\n\n\ndef _clamp(value: float, minimum: float, maximum: float) -> float:\n    return max(minimum, min(maximum, value))\n\n\ndef _create_job_dir(jobs_root: Path) -> Path:\n    stamp = time.strftime("%Y%m%d_%H%M%S")\n    jobs_root.mkdir(parents=True, exist_ok=True)\n    for counter in range(1000):\n        suffix = "" if counter == 0 else f"_{counter:03d}"\n        job_dir = jobs_root / f"segment_retime_{stamp}{suffix}"\n        try:\n            job_dir.mkdir(parents=True, exist_ok=False)\n            for child in ["input", "work", "output", "logs"]:\n                (job_dir / child).mkdir(parents=True, exist_ok=True)\n            return job_dir\n        except FileExistsError:\n            continue\n    raise RuntimeError(f"Could not create unique segment retime job folder under: {jobs_root}")\n\n\ndef _setup_logger(log_path: Path) -> logging.Logger:\n    logger = logging.getLogger(f"dichvideo.segment_retimer.{log_path.parent.parent.name}")\n    logger.setLevel(logging.INFO)\n    logger.handlers.clear()\n    formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")\n    file_handler = logging.FileHandler(log_path, encoding="utf-8")\n    file_handler.setFormatter(formatter)\n    logger.addHandler(file_handler)\n    stream_handler = logging.StreamHandler()\n    stream_handler.setFormatter(formatter)\n    logger.addHandler(stream_handler)\n    return logger\n\n\ndef _run(cmd: list[str], logger: logging.Logger) -> subprocess.CompletedProcess:\n    logger.info("Running command: %s", " ".join(cmd))\n    completed = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")\n    if completed.stdout.strip():\n        logger.info("stdout: %s", completed.stdout.strip()[-3000:])\n    if completed.stderr.strip():\n        logger.info("stderr: %s", completed.stderr.strip()[-3000:])\n    if completed.returncode != 0:\n        raise RuntimeError(f"Command failed with code {completed.returncode}: {\' \'.join(cmd)}")\n    return completed\n\n\ndef _check_binary(name: str) -> None:\n    if shutil.which(name) is None:\n        raise RuntimeError(f"Missing dependency: {name}. Hay cai ffmpeg va them vao PATH.")\n\n\ndef _write_json(path: Path, data) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")\n\n\ndef _tail(path: Path, lines: int = 160) -> str:\n    if not path.exists():\n        return ""\n    return "\\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:])\n', encoding='utf-8')


In [ ]:
from pathlib import Path
import re
import shutil
import sys
from google.colab import files

sys.path.insert(0, '/content')

from dichvideo.segment_retimer import SegmentRetimeConfig, retime_video_to_audio_segments

# Auto Sync SRT + Audio Segments settings.
# AUDIO_SEGMENTS_FOLDER_NAME='segments' uses OmniVoice clips and lets the sync step retime video/audio.
AUDIO_SEGMENTS_FOLDER_NAME = 'segments'
MIN_VIDEO_RATIO = 0.25
MAX_VIDEO_RATIO = 3.0
MIN_AUDIO_TEMPO = 0.75
MAX_AUDIO_TEMPO = 1.8
KEEP_GAPS = True
ORIGINAL_AUDIO_VOLUME = 0.0
CRF = 20
PRESET = 'veryfast'
OUTPUT_FPS = 30

VIDEO_EXTENSIONS = {'.mp4', '.mov', '.mkv', '.webm', '.avi'}
AUDIO_EXTENSIONS = {'.wav', '.mp3', '.m4a', '.flac', '.ogg'}

def safe_stem(path: Path) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', path.stem).strip('._') or 'srt'

def match_key(path: Path) -> str:
    return safe_stem(path).lower()

def srt_episode_number(srt_path: Path) -> int:
    stem = re.sub(r'_(vi|vn)$', '', srt_path.stem, flags=re.IGNORECASE)
    match = re.search(r'\((\d+)\)$', stem)
    return int(match.group(1)) + 1 if match else 1

def video_number(video_path: Path):
    match = re.fullmatch(r'(\d+)', video_path.stem.strip())
    return int(match.group(1)) if match else None

def match_video_for_srt(srt_path: Path, video_files: list[Path]) -> Path:
    if len(video_files) == 1:
        return video_files[0]

    target_number = srt_episode_number(srt_path)
    numbered_videos = {number: video for video in video_files if (number := video_number(video)) is not None}
    if target_number in numbered_videos:
        return numbered_videos[target_number]

    srt_key = match_key(srt_path)
    by_key = {match_key(video): video for video in video_files}
    if srt_key in by_key:
        return by_key[srt_key]
    candidates = [video for key, video in by_key.items() if srt_key in key or key in srt_key]
    if len(candidates) == 1:
        return candidates[0]
    raise RuntimeError(
        f'Cannot match video for {srt_path.name}. Supported pattern: base.srt -> 1.mp4, base(1).srt -> 2.mp4, base(2)_vi.srt -> 3.mp4.'
    )

video_files = sorted(path for path in VIDEO_DIR.iterdir() if path.suffix.lower() in VIDEO_EXTENSIONS)
if not video_files:
    print('Upload original video file(s) now for auto Sync SRT + Audio Segments.')
    uploaded_videos = files.upload()
    for name, data in uploaded_videos.items():
        if Path(name).suffix.lower() in VIDEO_EXTENSIONS:
            (VIDEO_DIR / name).write_bytes(data)
    video_files = sorted(path for path in VIDEO_DIR.iterdir() if path.suffix.lower() in VIDEO_EXTENSIONS)

srt_files = sorted(SRT_DIR.glob('*.srt'))
if not video_files:
    print('No uploaded video found. Skipping auto Sync SRT + Audio Segments.')
else:
    config = SegmentRetimeConfig(
        min_video_ratio=float(MIN_VIDEO_RATIO),
        max_video_ratio=float(MAX_VIDEO_RATIO),
        min_audio_tempo=float(MIN_AUDIO_TEMPO),
        max_audio_tempo=float(MAX_AUDIO_TEMPO),
        keep_gaps=bool(KEEP_GAPS),
        original_audio_volume=float(ORIGINAL_AUDIO_VOLUME),
        crf=int(CRF),
        preset=str(PRESET),
        output_fps=int(OUTPUT_FPS),
    )
    shutil.rmtree(FINAL_VIDEO_DIR, ignore_errors=True)
    FINAL_VIDEO_DIR.mkdir(parents=True, exist_ok=True)
    final_paths = []

    for srt_path in srt_files:
        video_path = match_video_for_srt(srt_path, video_files)
        audio_dir = OUTPUT_DIR / safe_stem(srt_path) / AUDIO_SEGMENTS_FOLDER_NAME
        if not audio_dir.exists():
            raise RuntimeError(f'Audio segment folder not found: {audio_dir}')
        audio_paths = sorted(
            [path for path in audio_dir.iterdir() if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS],
            key=lambda p: (int(re.findall(r'\d+', p.stem)[-1]) if re.findall(r'\d+', p.stem) else 10**9, p.name.lower())
        )
        if not audio_paths:
            raise RuntimeError(f'No audio segment files found: {audio_dir}')

        sync_srt_path = OUTPUT_DIR / safe_stem(srt_path) / f'{safe_stem(srt_path)}_grouped.srt'
        if not sync_srt_path.exists():
            sync_srt_path = srt_path

        print(f'Syncing {video_path.name} + {sync_srt_path.name} + {len(audio_paths)} audio segment(s)...')
        last_update = None
        for update in retime_video_to_audio_segments(video_path, sync_srt_path, audio_paths, SYNC_JOBS_DIR, config):
            last_update = update
            print(update.status)
        if not last_update or not last_update.video_path:
            raise RuntimeError(f'Sync failed for {srt_path.name}. Log: {last_update.log_path if last_update else "unknown"}')

        final_name = f'{safe_stem(srt_path)}_final.mp4'
        final_path = FINAL_VIDEO_DIR / final_name
        shutil.copy2(last_update.video_path, final_path)
        if last_update.schedule_path:
            shutil.copy2(last_update.schedule_path, FINAL_VIDEO_DIR / f'{safe_stem(srt_path)}_retime_schedule.json')
        if last_update.log_path:
            shutil.copy2(last_update.log_path, FINAL_VIDEO_DIR / f'{safe_stem(srt_path)}_segment_retimer.log')
        final_paths.append(final_path)
        print('Final video:', final_path)

    zip_base = FINAL_VIDEO_DIR.parent / 'dichvideo_final_videos'
    if zip_base.with_suffix('.zip').exists():
        zip_base.with_suffix('.zip').unlink()
    shutil.make_archive(str(zip_base), 'zip', FINAL_VIDEO_DIR)
    print('Created final videos zip:', zip_base.with_suffix('.zip'))


In [ ]:
from google.colab import files
from pathlib import Path

if Path('/content/dichvideo_final_videos.zip').exists():
    files.download('/content/dichvideo_final_videos.zip')
else:
    print('No final video zip found.')

if Path('/content/omnivoice_audio_results.zip').exists():
    files.download('/content/omnivoice_audio_results.zip')
else:
    print('No OmniVoice audio zip found.')
